# Candidate SSE Socio-Geodemographic Association

## Purpose

Test whether socio-geodemographic variables are associated with candidate SSE node membership. The notebook keeps data preparation explicit and delegates model fitting, odds-ratio tables, Wald tests, and fit statistics to `sse_detection.lib` (`sselib`).

## Inputs

- `sse_detection/sse_outputs/node_stats.parquet`
- `data/processed/scotland_clustering_analysis_dataset.parquet`
- Shared modelling helpers in `sse_detection/lib/regression.py`

## Outputs

- Composition and node-level mixing association tables in `sse_detection/association_outputs`
- Manuscript-facing figures and summary tables in `sse_detection/figures`

## Analysis Families

1. **Composition models** use sequence-window rows joined to node-level candidate status. These ask whether sequences in candidate nodes differ in age, sex, SIMD, urban/rural class, or health board composition from eligible background nodes.
2. **Node-level diversity/mixing models** use node-level entropy z-scores from `node_stats`. These ask whether candidate nodes are more or less mixed than expected for a node of the same size in the same window.

Eligible background nodes are restricted to `cluster_size >= min(candidate cluster size)`, so candidates are not compared against all singletons.

In [17]:
from pathlib import Path
from typing import Any
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils import load_analysis_columns  # noqa: E402
from sse_detection import lib as sselib  # noqa: E402E402

OUTPUT_DIR = PROJECT_ROOT / "sse_detection" / "sse_outputs"
RESULT_DIR = PROJECT_ROOT / "sse_detection" / "association_outputs"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)

## Model Specification

Primary models use Firth-penalised logistic regression with window and variant adjustment:

- `C(window_idx)`
- `C(clade)` by default, via `VARIANT_ADJUSTER`

Exact conditional logistic regression stratified by `window_idx` was considered, but the sequence-level window strata are too large for the recursive exact likelihood implementation in `statsmodels`. Firth penalisation is therefore used to reduce sparse-data/separation bias while retaining explicit window adjustment.

The window-level surveillance summaries (`wn_prop_sequenced`, `wn_positive_tests`) are documented below as an optional no-window-fixed-effect sensitivity specification. They are not included in the main primary models because they are constants within `window_idx`, so adding them alongside `C(window_idx)` creates rank collinearity and can prevent convergence.

Expanded composition models add each sequence's own standardised data-zone surveillance and epidemic-burden context. Expanded node-level models add node-level cluster means of the same context variables, standardised on the node analysis frame. Node-level mixing models use entropy null-model z-scores, so coefficients describe higher-than-window/size-expected mixing rather than absolute observed diversity.

Benjamini-Hochberg correction is applied within each displayed `domain` x `model_set` x `predictor_set` family. The confirmatory family is the single-predictor primary model set; expanded and joint models are sensitivity/robustness families and should not be pooled with the primary confirmatory family.


In [18]:
VARIANT_ADJUSTER = "clade"  # change to "who_voc" for a coarser variant adjustment
CLUSTER_SE = "cluster_id"  # reporting / sensitivity robust-SE clustering unit
WINDOW_STRATA = "window_idx"
MODEL_METHOD = "firth_glm"
MIXING_REFERENCE = "per 1 SD entropy null-model z-score"

PRIMARY_COMPOSITION_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

WINDOW_SURVEILLANCE_ADJUSTERS = [
    "z_wn_prop_sequenced",
    "z_log1p_wn_positive_tests",
]

# Optional sensitivity model if you want window-level surveillance adjustment
# instead of window fixed effects. Do not combine these with C(window_idx).
SURVEILLANCE_COMPOSITION_ADJUSTERS = [
    f"C({VARIANT_ADJUSTER})",
    *WINDOW_SURVEILLANCE_ADJUSTERS,
]

EXPANDED_CONTEXT_ADJUSTERS = [
    "z_dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests",
]

EXPANDED_COMPOSITION_ADJUSTERS = (
    PRIMARY_COMPOSITION_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS
)

PRIMARY_MIXING_ADJUSTERS = [
    "C(window_idx)",
    f"C({VARIANT_ADJUSTER})",
]

EXPANDED_MIXING_ADJUSTERS = PRIMARY_MIXING_ADJUSTERS + EXPANDED_CONTEXT_ADJUSTERS

COMPOSITION_MODEL_SETS = {
    "primary": PRIMARY_COMPOSITION_ADJUSTERS,
    "expanded": EXPANDED_COMPOSITION_ADJUSTERS,
    # "surveillance_no_window_fe": SURVEILLANCE_COMPOSITION_ADJUSTERS,
}

MIXING_MODEL_SETS = {
    "primary": PRIMARY_MIXING_ADJUSTERS,
    "expanded": EXPANDED_MIXING_ADJUSTERS,
}

COMPOSITION_SPECS = [
    {
        "name": "sex",
        "column": "sex",
        "reference": "Male",
        "label": "Sex",
    },
    {
        "name": "age_band",
        "column": "age_band",
        "reference": "30-34",
        "fallback_references": ["35-39", "25-29"],
        "label": "Age band",
    },
    {
        "name": "simd_quintile",
        "column": "dz_simd_quintile",
        "reference": "3",
        "label": "SIMD quintile",
    },
    {
        "name": "urban_rural_class",
        "column": "dz_urban_rural_class",
        "reference": "Large Urban Areas",
        "label": "Urban/rural class",
    },
    {
        "name": "health_board",
        "column": "dz_health_board",
        "reference": "Greater Glasgow and Clyde",
        "label": "Health board",
    },
]

MIXING_FEATURES = [
    "sex_entropy_z",
    "age_entropy_z",
    "simd_entropy_z",
    "urban_rural_entropy_z",
    "health_board_entropy_z",
]

STANDARDISE_SPECS = {
    "z_wn_prop_sequenced": "wn_prop_sequenced",
    "z_log1p_wn_positive_tests": "log1p_wn_positive_tests",
    "z_dz_cum_prop_sequenced": "dz_cum_prop_sequenced",
    "z_dz_cum_incidence_per_capita": "dz_cum_incidence_per_capita",
    "z_dz_7d_test_positivity": "dz_7d_test_positivity",
    "z_log1p_dz_cum_positive_tests": "log1p_dz_cum_positive_tests",
}


def add_standardised_adjusters(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    if "wn_positive_tests" in out.columns:
        out["log1p_wn_positive_tests"] = np.log1p(out["wn_positive_tests"])
    if "dz_cum_positive_tests" in out.columns:
        out["log1p_dz_cum_positive_tests"] = np.log1p(out["dz_cum_positive_tests"])
    for target, source in STANDARDISE_SPECS.items():
        if source not in out.columns:
            continue
        values = out[source].astype(float)
        sd = values.std(skipna=True)
        if pd.isna(sd) or sd == 0:
            out[target] = np.nan
        else:
            out[target] = (values - values.mean(skipna=True)) / sd
    return out

## Load and Prepare Analysis Frames

`node_stats` is the source of the candidate label and the node-level mixing metrics. For composition models, sequence-window rows are loaded from the processed analysis dataset using `utils.data`, filtered to match `sse_detection.ipynb`'s retained odd windows, renumbered, and then joined to eligible node status.

In [20]:
outs = sselib.load_sse_outputs(OUTPUT_DIR)
node_stats = outs.node_stats.copy()

min_candidate_size = int(
    node_stats.loc[node_stats["sse_candidate"], "cluster_size"].min()
)
eligible_nodes = node_stats.loc[
    node_stats["cluster_size"].ge(min_candidate_size)
].copy()

if CLUSTER_SE not in eligible_nodes.columns:
    raise KeyError(f"{CLUSTER_SE!r} is not present in node_stats.")
if VARIANT_ADJUSTER not in eligible_nodes.columns:
    raise KeyError(f"{VARIANT_ADJUSTER!r} is not present in node_stats.")

# Node-level frame for mixing models.
node_model_base = eligible_nodes.copy()
node_model_base["candidate"] = node_model_base["sse_candidate"].astype(int)
node_model_base[VARIANT_ADJUSTER] = (
    node_model_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
)
node_model_base[CLUSTER_SE] = (
    node_model_base[CLUSTER_SE].fillna(node_model_base["cluster_id"]).astype(str)
)
node_model_base = add_standardised_adjusters(node_model_base)

node_key = eligible_nodes[
    ["meta_cluster_id", "sse_candidate", CLUSTER_SE, "cluster_size"]
].drop_duplicates("cluster_id")

sequence_columns = sorted(
    {
        "window_id",
        "window_idx",
        "cluster_id",
        "sequence_id",
        "clade",
        "who_voc",
        "wn_prop_sequenced",
        "wn_positive_tests",
        "dz_cum_prop_sequenced",
        "dz_cum_incidence_per_capita",
        "dz_7d_test_positivity",
        "dz_cum_positive_tests",
        *(spec["column"] for spec in COMPOSITION_SPECS),
    }
)

sequence_raw = load_analysis_columns(
    sequence_columns,
    add_policy=False,
    window_stride=2,
)


composition_base = sequence_raw.merge(node_key, on="cluster_id", how="inner")
composition_base["candidate"] = composition_base["sse_candidate"].astype(int)
composition_base[VARIANT_ADJUSTER] = (
    composition_base[VARIANT_ADJUSTER].fillna("Missing").astype(str)
)
composition_base[CLUSTER_SE] = (
    composition_base[CLUSTER_SE].fillna(composition_base["cluster_id"]).astype(str)
)
composition_base = add_standardised_adjusters(composition_base)

print(f"node_stats: {len(node_stats):,} nodes")
print(f"candidate nodes: {node_stats['sse_candidate'].sum():,}")
print(f"minimum candidate cluster size: {min_candidate_size}")
print(f"eligible nodes: {len(eligible_nodes):,}")
print(f"eligible candidates: {eligible_nodes['sse_candidate'].sum():,}")
print(f"eligible background: {(~eligible_nodes['sse_candidate']).sum():,}")
print()
print(f"composition sequence-window rows: {len(composition_base):,}")
print(
    f"unique sequences in composition frame: {composition_base['sequence_id'].nunique():,}"
)
print(f"nodes in composition frame: {composition_base['cluster_id'].nunique():,}")
print(
    f"candidate share in composition frame: {composition_base['candidate'].mean():.1%}"
)

cluster_diagnostics = pd.concat(
    [
        sselib.cluster_se_diagnostics(
            composition_base,
            CLUSTER_SE,
            outcome="candidate",
        ).assign(analysis_frame="composition"),
        sselib.cluster_se_diagnostics(
            node_model_base,
            CLUSTER_SE,
            outcome="candidate",
        ).assign(analysis_frame="node_mixing"),
    ],
    ignore_index=True,
)
display(
    cluster_diagnostics[
        [
            "analysis_frame",
            "cluster_col",
            "n_rows",   
            "n_clusters",
            "outcome_positive_clusters",
            "outcome_varying_clusters",
            "min_rows_per_cluster",
            "median_rows_per_cluster",
        ]
    ]
)


node_stats: 99,006 nodes
candidate nodes: 6,907
minimum candidate cluster size: 6
eligible nodes: 13,059
eligible candidates: 6,907
eligible background: 6,152

composition sequence-window rows: 264,139
unique sequences in composition frame: 188,106
nodes in composition frame: 13,059
candidate share in composition frame: 55.8%


,analysis_frame,cluster_col,n_rows,n_clusters,outcome_positive_clusters,outcome_varying_clusters,min_rows_per_cluster,median_rows_per_cluster
0,composition,cluster_id,264139,13059,6907,0,1,9.0
1,node_mixing,cluster_id,13059,13059,6907,0,1,1.0


## Preparation Helpers

The functions below deliberately handle data preparation in the notebook: complete-case filtering, reference-level resolution, and removal of window strata with no candidate/background variation. The regression module is then called only on already-prepared model frames.

Primary models are Firth-penalised logistic regressions with explicit `C(window_idx)` adjustment. Dropped row and dropped stratum counts are carried into the exported tables so sparse-window filtering is visible in the results.


In [21]:
def fit_association_result(data: pd.DataFrame, formula: str):
    if MODEL_METHOD == "conditional_logit_by_window":
        return sselib.fit_conditional_logit(
            data,
            formula,
            strata_col=WINDOW_STRATA,
        )
    if MODEL_METHOD == "firth_glm":
        return sselib.fit_firth_logit(data, formula)
    if MODEL_METHOD == "glm_clustered":
        return sselib.fit_binomial_glm(data, formula, cluster_col=CLUSTER_SE)
    raise ValueError(f"Unknown MODEL_METHOD={MODEL_METHOD!r}")


def fit_exposure_association(
    data: pd.DataFrame,
    *,
    outcome: str,
    exposure: str,
    adjusters: list[str],
    model_name: str,
    reference=None,
    categorical: bool = True,
) -> sselib.AssociationModel:
    exposure_term = (
        sselib.categorical_term(exposure, reference=reference)
        if categorical
        else exposure
    )
    formula = sselib.make_formula(outcome, exposure_term, adjusters)
    result = fit_association_result(data, formula)
    odds = sselib.tidy_odds_ratios(
        result,
        model_name=model_name,
        term_filter=exposure_term,
    )
    if categorical:
        wald = sselib.robust_wald_for_prefix(
            result,
            exposure_term,
            model_name=model_name,
            term=exposure,
        )
    else:
        wald = sselib.tidy_single_parameter_wald(
            result,
            [exposure],
            model_name=model_name,
        )
    wald["formula"] = formula
    return sselib.AssociationModel(
        result=result,
        odds_ratios=odds,
        wald=wald,
        formula=formula,
    )


def resolve_reference(data: pd.DataFrame, column: str, preferred, fallbacks=None):
    levels = set(data[column].dropna().astype(str))
    candidates = [preferred, *(fallbacks or [])]
    for ref in candidates:
        if ref is None:
            continue
        ref_str = str(ref)
        if ref_str in levels:
            return ref_str
        for level in levels:
            try:
                if float(level) == float(ref_str):
                    return level
            except ValueError:
                pass
    counts = data[column].dropna().astype(str).value_counts()
    if counts.empty:
        raise ValueError(f"No observed levels for {column!r}.")
    return str(counts.index[0])


def complete_case(data: pd.DataFrame, required: list[str]) -> pd.DataFrame:
    missing = [col for col in required if col not in data.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")
    return data.dropna(subset=required).copy()


def drop_nonvarying_levels(
    data: pd.DataFrame,
    columns: list[str],
    *,
    outcome: str = "candidate",
) -> tuple[pd.DataFrame, dict[str, int], dict[str, int]]:
    d = data.copy()
    dropped_rows: dict[str, int] = {}
    dropped_strata: dict[str, int] = {}
    changed = True
    while changed:
        changed = False
        for col in columns:
            if col not in d.columns:
                continue
            strata_nunique = d.groupby(col, dropna=False)[outcome].transform("nunique")
            varies = strata_nunique.gt(1)
            n_drop = int((~varies).sum())
            if n_drop:
                dropped_rows[col] = dropped_rows.get(col, 0) + n_drop
                dropped_strata[col] = dropped_strata.get(col, 0) + int(
                    d.loc[~varies, col].nunique(dropna=False)
                )
                d = d.loc[varies].copy()
                changed = True
    return d, dropped_rows, dropped_strata


def dropped_metadata(
    dropped_rows: dict[str, int], dropped_strata: dict[str, int]
) -> dict[str, object]:
    return {
        "dropped_nonvarying_rows": sum(dropped_rows.values()),
        "dropped_nonvarying_strata": sum(dropped_strata.values()),
        "dropped_nonvarying_detail": repr(
            {
                "rows": dropped_rows,
                "strata": dropped_strata,
            }
        ),
    }


def add_model_metadata(table: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = table.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def add_fit_metadata(fit_stats: pd.DataFrame, **metadata) -> pd.DataFrame:
    out = fit_stats.copy()
    for key, value in reversed(list(metadata.items())):
        out.insert(0, key, value)
    return out


def bh_adjust_by(
    table: pd.DataFrame, group_cols: list[str], p_col: str = "P>chi2"
) -> pd.DataFrame:
    if table.empty:
        return table.copy()
    out = table.copy()
    out["p_adj_bh"] = np.nan
    for _, idx in out.groupby(group_cols, dropna=False).groups.items():
        adjusted = sselib.bh_adjust(out.loc[idx], p_col=p_col)
        out.loc[idx, "p_adj_bh"] = adjusted["p_adj_bh"].to_numpy()
    return out

## Composition Models

Each composition predictor is first entered **one at a time**. Then all composition predictors are entered **together**. Both sets are fitted with primary adjusters and expanded adjusters. Odds ratios are sequence-level associations with candidate-node membership, with robust standard errors clustered by `meta_cluster_id`.

In [22]:
def prepare_composition_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", "sequence_id", CLUSTER_SE, WINDOW_STRATA]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(composition_base, required)
    for col in predictors:
        d[col] = d[col].astype(str)
    strata = [WINDOW_STRATA]
    d, dropped_rows, dropped_strata = drop_nonvarying_levels(d, strata)
    return d, dropped_rows, dropped_strata


def fit_single_composition_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for spec in COMPOSITION_SPECS:
        predictor = spec["column"]
        d, dropped_rows, dropped_strata = prepare_composition_frame(
            [predictor], adjusters
        )
        reference = resolve_reference(
            d,
            predictor,
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        model_name = f"composition__{model_set}__single__{spec['name']}"
        fit = fit_exposure_association(
            d,
            outcome="candidate",
            exposure=predictor,
            adjusters=adjusters,
            model_name=model_name,
            reference=reference,
            categorical=True,
        )
        fitted[spec["name"]] = fit.result

        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": reference,
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            **dropped_metadata(dropped_rows, dropped_strata),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(add_model_metadata(fit.odds_ratios, **meta))
        fit_tables.append(
            add_fit_metadata(
                sselib.model_fit_stats(
                    fit.result, model_name=model_name, formula=fit.formula
                ),
                **meta,
            )
        )
        print(f"Fitted {model_name}: {len(d):,} rows", flush=True)

    return (
        fitted,
        pd.concat(wald_tables, ignore_index=True),
        pd.concat(or_tables, ignore_index=True),
        pd.concat(fit_tables, ignore_index=True),
    )


def fit_joint_composition_model(model_set: str, adjusters: list[str]) -> Any:
    predictors = [spec["column"] for spec in COMPOSITION_SPECS]
    d, dropped_rows, dropped_strata = prepare_composition_frame(predictors, adjusters)

    terms = []
    references = {}
    for spec in COMPOSITION_SPECS:
        reference = resolve_reference(
            d,
            spec["column"],
            spec.get("reference"),
            spec.get("fallback_references"),
        )
        references[spec["name"]] = reference
        terms.append(sselib.categorical_term(spec["column"], reference))

    formula = "candidate ~ " + " + ".join(terms + adjusters)
    model_name = f"composition__{model_set}__joint"
    result = fit_association_result(d, formula)

    wald_tables = []
    for spec, term in zip(COMPOSITION_SPECS, terms):
        wald = sselib.robust_wald_for_prefix(
            result,
            term,
            model_name=model_name,
            term=spec["name"],
        )
        meta = {
            "domain": "composition",
            "model_set": model_set,
            "predictor_set": "joint",
            "predictor": spec["name"],
            "label": spec["label"],
            "reference": references[spec["name"]],
            "n_model_rows": len(d),
            "n_sequences": d["sequence_id"].nunique(),
            "n_nodes": d["cluster_id"].nunique(),
            **dropped_metadata(dropped_rows, dropped_strata),
        }
        wald_tables.append(add_model_metadata(wald, **meta))

    term_names = [
        name for term in terms for name in sselib.parameter_names_for_term(result, term)
    ]
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].isin(term_names)].copy()
    odds = add_model_metadata(
        odds,
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        **dropped_metadata(dropped_rows, dropped_strata),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="composition",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_composition",
        label="All composition predictors",
        reference=repr(references),
        n_model_rows=len(d),
        n_sequences=d["sequence_id"].nunique(),
        n_nodes=d["cluster_id"].nunique(),
        **dropped_metadata(dropped_rows, dropped_strata),
    )
    print(f"Fitted {model_name}: {len(d):,} rows", flush=True)
    return result, pd.concat(wald_tables, ignore_index=True), odds, fit_stats


composition_model_results = {}
composition_fits = {}


def run_composition_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = (
        fit_single_composition_models(model_set, adjusters)
    )
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_composition_model(
        model_set, adjusters
    )
    composition_fits[(model_set, "single")] = single_fit
    composition_fits[(model_set, "joint")] = joint_fit
    composition_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return composition_model_results[model_set]

### Primary Composition Models

These are the main single-predictor and joint composition models adjusted for window, variant, and window-level surveillance intensity.

In [23]:
composition_primary_results = run_composition_model_set(
    "primary",
    COMPOSITION_MODEL_SETS["primary"],
)

composition_primary_results["wald"][
    [
        "domain",
        "model_set",
        "predictor_set",
        "predictor",
        "label",
        "reference",
        "chi2",
        "df",
        "P>chi2",
        "n_model_rows",
        "n_sequences",
        "n_nodes",
        "dropped_nonvarying_detail",
    ]
]

Fitted composition__primary__single__sex: 264,111 rows
Fitted composition__primary__single__age_band: 264,111 rows
Fitted composition__primary__single__simd_quintile: 264,111 rows
Fitted composition__primary__single__urban_rural_class: 264,111 rows
Fitted composition__primary__single__health_board: 264,111 rows
Fitted composition__primary__joint: 264,111 rows


,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_detail
0,composition,primary,single,sex,Sex,Male,2.436386,1,1.185491e-01,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
1,composition,primary,single,age_band,Age band,30-34,22.743596,15,8.973759e-02,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
2,composition,primary,single,simd_quintile,SIMD quintile,3,3.868955,4,4.240313e-01,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
3,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,91.060703,5,4.022191e-18,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
4,composition,primary,single,health_board,Health board,Greater Glasgow and Clyde,119.197338,13,2.900509e-19,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
5,composition,primary,joint,sex,Sex,Male,2.561533,1,1.094924e-01,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
6,composition,primary,joint,age_band,Age band,30-34,22.667661,15,9.145547e-02,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
7,composition,primary,joint,simd_quintile,SIMD quintile,3,2.320146,4,6.771037e-01,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
8,composition,primary,joint,urban_rural_class,Urban/rural class,Large Urban Areas,48.580552,5,2.703415e-09,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
9,composition,primary,joint,health_board,Health board,Greater Glasgow and Clyde,78.455095,13,2.152118e-11,264111,188104,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."


### Expanded Composition Models

These repeat the composition models after adding sequence data-zone surveillance and epidemic-burden context. Treat these as a robustness/sensitivity specification.

In [24]:
composition_expanded_results = run_composition_model_set(
    "expanded",
    COMPOSITION_MODEL_SETS["expanded"],
)

composition_expanded_results["wald"][
    [
        "domain",
        "model_set",
        "predictor_set",
        "predictor",
        "label",
        "reference",
        "chi2",
        "df",
        "P>chi2",
        "n_model_rows",
        "n_sequences",
        "n_nodes",
        "dropped_nonvarying_detail",
    ]
]

Fitted composition__expanded__single__sex: 264,091 rows
Fitted composition__expanded__single__age_band: 264,091 rows
Fitted composition__expanded__single__simd_quintile: 264,091 rows
Fitted composition__expanded__single__urban_rural_class: 264,091 rows
Fitted composition__expanded__single__health_board: 264,091 rows
Fitted composition__expanded__joint: 264,091 rows


,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_detail
0,composition,expanded,single,sex,Sex,Male,1.846406,1,1.742020e-01,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
1,composition,expanded,single,age_band,Age band,30-34,24.688806,15,5.428200e-02,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
2,composition,expanded,single,simd_quintile,SIMD quintile,3,33.288180,4,1.042708e-06,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
3,composition,expanded,single,urban_rural_class,Urban/rural class,Large Urban Areas,375.659947,5,5.211393e-79,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
4,composition,expanded,single,health_board,Health board,Greater Glasgow and Clyde,608.439599,13,1.216826e-121,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
5,composition,expanded,joint,sex,Sex,Male,2.221140,1,1.361325e-01,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
6,composition,expanded,joint,age_band,Age band,30-34,23.676310,15,7.079816e-02,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
7,composition,expanded,joint,simd_quintile,SIMD quintile,3,11.730574,4,1.947139e-02,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
8,composition,expanded,joint,urban_rural_class,Urban/rural class,Large Urban Areas,136.649120,5,9.220379e-28,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
9,composition,expanded,joint,health_board,Health board,Greater Glasgow and Clyde,371.113357,13,2.780703e-71,264091,188091,13058,"{'rows': {'window_idx': 28}, 'strata': {'windo..."


### Composition Summary Tables

The omnibus Wald table is the main screening table. McFadden pseudo-R2 is included for relative comparison within this model family.

In [25]:
if not composition_model_results:
    raise RuntimeError(
        "Run at least one composition model-set cell before summarising."
    )

composition_wald = pd.concat(
    [result["wald"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_wald = bh_adjust_by(
    composition_wald, ["domain", "model_set", "predictor_set"]
)
composition_or = pd.concat(
    [result["odds"] for result in composition_model_results.values()],
    ignore_index=True,
)
composition_fit_stats = pd.concat(
    [result["fit_stats"] for result in composition_model_results.values()],
    ignore_index=True,
)

display(
    composition_wald[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "label",
            "reference",
            "chi2",
            "df",
            "P>chi2",
            "p_adj_bh",
            "n_model_rows",
            "n_sequences",
            "n_nodes",
            "dropped_nonvarying_rows",
            "dropped_nonvarying_strata",
            "dropped_nonvarying_detail",
        ]
    ]
)

display(
    composition_fit_stats[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "r2_mcfadden",
            "converged",
            "aic",
            "bic_llf",
            "log_likelihood",
            "ll_null",
            "n_model_rows",
            "n_sequences",
            "n_nodes",
        ]
    ]
)

,domain,model_set,predictor_set,predictor,label,reference,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_sequences,n_nodes,dropped_nonvarying_rows,dropped_nonvarying_strata,dropped_nonvarying_detail
0,composition,primary,single,sex,Sex,Male,2.436386,1,1.185491e-01,1.481863e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
1,composition,primary,single,age_band,Age band,30-34,22.743596,15,8.973759e-02,1.481863e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
2,composition,primary,single,simd_quintile,SIMD quintile,3,3.868955,4,4.240313e-01,4.240313e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
3,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,91.060703,5,4.022191e-18,1.005548e-17,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
4,composition,primary,single,health_board,Health board,Greater Glasgow and Clyde,119.197338,13,2.900509e-19,1.450255e-18,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
5,composition,primary,joint,sex,Sex,Male,2.561533,1,1.094924e-01,1.368655e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
6,composition,primary,joint,age_band,Age band,30-34,22.667661,15,9.145547e-02,1.368655e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
7,composition,primary,joint,simd_quintile,SIMD quintile,3,2.320146,4,6.771037e-01,6.771037e-01,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
8,composition,primary,joint,urban_rural_class,Urban/rural class,Large Urban Areas,48.580552,5,2.703415e-09,6.758537e-09,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."
9,composition,primary,joint,health_board,Health board,Greater Glasgow and Clyde,78.455095,13,2.152118e-11,1.076059e-10,264111,188104,13058,28,1,"{'rows': {'window_idx': 28}, 'strata': {'windo..."


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_sequences,n_nodes
0,composition,primary,single,sex,NaN,True,335202.486013,NaN,-167514.243006,NaN,264111,188104,13058
1,composition,primary,single,age_band,NaN,True,335210.145935,NaN,-167504.072968,NaN,264111,188104,13058
2,composition,primary,single,simd_quintile,NaN,True,335207.052527,NaN,-167513.526263,NaN,264111,188104,13058
3,composition,primary,single,urban_rural_class,NaN,True,335121.734923,NaN,-167469.867461,NaN,264111,188104,13058
4,composition,primary,single,health_board,NaN,True,335108.982902,NaN,-167455.491451,NaN,264111,188104,13058
5,composition,primary,joint,all_composition,NaN,True,335081.949559,NaN,-167416.974780,NaN,264111,188104,13058
6,composition,expanded,single,sex,NaN,True,334141.765136,NaN,-166979.882568,NaN,264091,188091,13058
7,composition,expanded,single,age_band,NaN,True,334146.863440,NaN,-166968.431720,NaN,264091,188091,13058
8,composition,expanded,single,simd_quintile,NaN,True,334116.282066,NaN,-166964.141033,NaN,264091,188091,13058
9,composition,expanded,single,urban_rural_class,NaN,True,333773.683351,NaN,-166791.841676,NaN,264091,188091,13058


### Composition Odds Ratios

The table below contains coefficient-level odds ratios for the composition models. For primary interpretation, start with the omnibus Wald table above; use this table to identify which levels drive an omnibus association.

In [26]:
display(
    composition_or[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "label",
            "reference",
            "term",
            "estimate",
            "std_error",
            "p_value",
            "odds_ratio",
            "or_low",
            "or_high",
        ]
    ].sort_values(["model_set", "predictor_set", "predictor", "p_value"])
)

,domain,model_set,predictor_set,predictor,label,reference,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
144,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.234152,0.018333,2.342159e-37,1.263837,1.219231,1.310075
137,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.271453,0.028612,2.372444e-21,1.311869,1.240325,1.387539
136,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_urban_rural_class, Treatment(reference='L...",0.111806,0.012382,1.718031e-19,1.118295,1.091484,1.145766
150,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.157251,0.018254,7.031602e-18,1.170289,1.129159,1.212918
145,composition,expanded,joint,all_composition,All composition predictors,"{'sex': 'Male', 'age_band': '30-34', 'simd_qui...","C(dz_health_board, Treatment(reference='Greate...",0.197217,0.025785,2.035606e-14,1.218008,1.157981,1.281146
...,...,...,...,...,...,...,...,...,...,...,...,...,...
22,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.077790,0.009563,4.146589e-16,1.080896,1.060825,1.101347
23,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.131394,0.023415,2.004242e-08,1.140417,1.089264,1.193972
24,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.118337,0.025736,4.263183e-06,1.125624,1.070254,1.183859
20,composition,primary,single,urban_rural_class,Urban/rural class,Large Urban Areas,"C(dz_urban_rural_class, Treatment(reference='L...",0.056616,0.014960,1.540703e-04,1.058249,1.027670,1.089738


## Node-Level Diversity / Mixing Models

Node-level mixing models use entropy null-model z-scores. Coefficients are reported per SD in the z-score, so they compare nodes that are more or less mixed than expected for their size and window. Primary models adjust for window and variant; expanded models additionally adjust for standardised node-level cluster means of data-zone surveillance and burden variables.


Do **not** add candidate-defining quantities such as `cluster_size`, `core_amplification_score`, `out_strength`, or `onward_dissemination_score` to the main models, because that would condition on the machinery used to define the outcome.

In [27]:
def prepare_mixing_frame(predictors: list[str], adjusters: list[str]) -> Any:
    required = (
        ["candidate", "cluster_id", CLUSTER_SE, WINDOW_STRATA]
        + predictors
        + sselib.model_variables_from_terms(adjusters)
    )
    required = list(dict.fromkeys(required))
    d = complete_case(node_model_base, required)
    strata = [WINDOW_STRATA]
    d, dropped_rows, dropped_strata = drop_nonvarying_levels(d, strata)
    return d, dropped_rows, dropped_strata


def fit_single_mixing_models(model_set: str, adjusters: list[str]) -> Any:
    wald_tables = []
    or_tables = []
    fit_tables = []
    fitted = {}

    for feature in MIXING_FEATURES:
        if feature not in node_model_base.columns:
            print(f"Skipping {feature}: not found", flush=True)
            continue
        d, dropped_rows, dropped_strata = prepare_mixing_frame([feature], adjusters)
        model_name = f"mixing__{model_set}__single__{feature}"
        fit = fit_exposure_association(
            d,
            outcome="candidate",
            exposure=feature,
            adjusters=adjusters,
            model_name=model_name,
            categorical=False,
        )
        fitted[feature] = fit.result
        meta = {
            "domain": "node_mixing",
            "model_set": model_set,
            "predictor_set": "single",
            "predictor": feature,
            "label": feature.replace("_", " "),
            "reference": MIXING_REFERENCE,
            "n_model_rows": len(d),
            "n_nodes": d["cluster_id"].nunique(),
            **dropped_metadata(dropped_rows, dropped_strata),
        }
        wald_tables.append(add_model_metadata(fit.wald, **meta))
        or_tables.append(
            add_model_metadata(
                fit.odds_ratios.loc[fit.odds_ratios["term"].eq(feature)].copy(),
                **meta,
            )
        )
        fit_tables.append(
            add_fit_metadata(
                sselib.model_fit_stats(
                    fit.result, model_name=model_name, formula=fit.formula
                ),
                **meta,
            )
        )
        print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return (
        fitted,
        pd.concat(wald_tables, ignore_index=True),
        pd.concat(or_tables, ignore_index=True),
        pd.concat(fit_tables, ignore_index=True),
    )


def fit_joint_mixing_model(model_set: str, adjusters: list[str]) -> Any:
    features = [
        feature for feature in MIXING_FEATURES if feature in node_model_base.columns
    ]
    d, dropped_rows, dropped_strata = prepare_mixing_frame(features, adjusters)
    formula = "candidate ~ " + " + ".join(features + adjusters)
    model_name = f"mixing__{model_set}__joint"
    result = fit_association_result(d, formula)

    wald = sselib.tidy_single_parameter_wald(result, features, model_name=model_name)
    wald = add_model_metadata(
        wald,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        **dropped_metadata(dropped_rows, dropped_strata),
    )
    odds = sselib.tidy_odds_ratios(result, model_name=model_name)
    odds = odds.loc[odds["term"].isin(features)].copy()
    odds = add_model_metadata(
        odds,
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        **dropped_metadata(dropped_rows, dropped_strata),
    )
    fit_stats = add_fit_metadata(
        sselib.model_fit_stats(result, model_name=model_name, formula=formula),
        domain="node_mixing",
        model_set=model_set,
        predictor_set="joint",
        predictor="all_mixing",
        label="All mixing predictors",
        reference=MIXING_REFERENCE,
        n_model_rows=len(d),
        n_nodes=d["cluster_id"].nunique(),
        **dropped_metadata(dropped_rows, dropped_strata),
    )
    print(f"Fitted {model_name}: {len(d):,} nodes", flush=True)
    return result, wald, odds, fit_stats

In [28]:
mixing_model_results = {}
mixing_fits = {}


def run_mixing_model_set(model_set: str, adjusters: list[str]) -> Any:
    single_fit, single_wald, single_or, single_fit_stats = fit_single_mixing_models(
        model_set, adjusters
    )
    joint_fit, joint_wald, joint_or, joint_fit_stats = fit_joint_mixing_model(
        model_set, adjusters
    )
    mixing_fits[(model_set, "single")] = single_fit
    mixing_fits[(model_set, "joint")] = joint_fit
    mixing_model_results[model_set] = {
        "wald": pd.concat([single_wald, joint_wald], ignore_index=True),
        "odds": pd.concat([single_or, joint_or], ignore_index=True),
        "fit_stats": pd.concat([single_fit_stats, joint_fit_stats], ignore_index=True),
    }
    return mixing_model_results[model_set]


### Primary Mixing Models

These node-level models test each entropy alone and then all entropies together, adjusted for window and variant.

In [29]:
mixing_primary_results = run_mixing_model_set(
    "primary",
    MIXING_MODEL_SETS["primary"],
)

mixing_primary_results["wald"][
    [
        "domain",
        "model_set",
        "predictor_set",
        "predictor",
        "term",
        "chi2",
        "df",
        "P>chi2",
        "n_model_rows",
        "n_nodes",
        "dropped_nonvarying_detail",
    ]
]

Fitted mixing__primary__single__sex_entropy_z: 12,966 nodes
Fitted mixing__primary__single__age_entropy_z: 12,966 nodes
Fitted mixing__primary__single__simd_entropy_z: 12,966 nodes
Fitted mixing__primary__single__urban_rural_entropy_z: 12,966 nodes
Fitted mixing__primary__single__health_board_entropy_z: 12,966 nodes
Fitted mixing__primary__joint: 12,966 nodes


,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,n_model_rows,n_nodes,dropped_nonvarying_detail
0,node_mixing,primary,single,sex_entropy_z,sex_entropy_z,1.650174,1,0.198935,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
1,node_mixing,primary,single,age_entropy_z,age_entropy_z,8.026412,1,0.004610,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
2,node_mixing,primary,single,simd_entropy_z,simd_entropy_z,2.020922,1,0.155145,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
3,node_mixing,primary,single,urban_rural_entropy_z,urban_rural_entropy_z,0.004173,1,0.948495,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
4,node_mixing,primary,single,health_board_entropy_z,health_board_entropy_z,1.951550,1,0.162420,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
5,node_mixing,primary,joint,all_mixing,sex_entropy_z,0.808100,1,0.368683,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
6,node_mixing,primary,joint,all_mixing,age_entropy_z,5.699678,1,0.016968,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
7,node_mixing,primary,joint,all_mixing,simd_entropy_z,0.369690,1,0.543173,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
8,node_mixing,primary,joint,all_mixing,urban_rural_entropy_z,0.726623,1,0.393980,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
9,node_mixing,primary,joint,all_mixing,health_board_entropy_z,0.757914,1,0.383982,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."


### Expanded Mixing Models

These add node-level cluster means of the data-zone context variables, matching the expanded composition sensitivity model.

In [30]:
mixing_expanded_results = run_mixing_model_set(
    "expanded",
    MIXING_MODEL_SETS["expanded"],
)

mixing_expanded_results["wald"][
    [
        "domain",
        "model_set",
        "predictor_set",
        "predictor",
        "term",
        "chi2",
        "df",
        "P>chi2",
        "n_model_rows",
        "n_nodes",
        "dropped_nonvarying_detail",
    ]
]

Fitted mixing__expanded__single__sex_entropy_z: 12,966 nodes
Fitted mixing__expanded__single__age_entropy_z: 12,966 nodes
Fitted mixing__expanded__single__simd_entropy_z: 12,966 nodes
Fitted mixing__expanded__single__urban_rural_entropy_z: 12,966 nodes
Fitted mixing__expanded__single__health_board_entropy_z: 12,966 nodes
Fitted mixing__expanded__joint: 12,966 nodes


,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,n_model_rows,n_nodes,dropped_nonvarying_detail
0,node_mixing,expanded,single,sex_entropy_z,sex_entropy_z,1.051956,1,0.305057,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
1,node_mixing,expanded,single,age_entropy_z,age_entropy_z,11.619283,1,0.000653,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
2,node_mixing,expanded,single,simd_entropy_z,simd_entropy_z,4.843455,1,0.027751,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
3,node_mixing,expanded,single,urban_rural_entropy_z,urban_rural_entropy_z,2.504361,1,0.113532,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
4,node_mixing,expanded,single,health_board_entropy_z,health_board_entropy_z,5.511410,1,0.018893,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
5,node_mixing,expanded,joint,all_mixing,sex_entropy_z,0.214822,1,0.643014,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
6,node_mixing,expanded,joint,all_mixing,age_entropy_z,8.188560,1,0.004216,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
7,node_mixing,expanded,joint,all_mixing,simd_entropy_z,1.838758,1,0.175097,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
8,node_mixing,expanded,joint,all_mixing,urban_rural_entropy_z,11.145277,1,0.000842,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."
9,node_mixing,expanded,joint,all_mixing,health_board_entropy_z,6.459909,1,0.011034,12966,12966,"{'rows': {'window_idx': 1}, 'strata': {'window..."


### Mixing Summary Tables

The Wald table gives the per-feature evidence. The odds-ratio table below gives the direction and magnitude per 0.1 increase in normalised entropy.

In [31]:
if not mixing_model_results:
    raise RuntimeError("Run at least one mixing model-set cell before summarising.")

mixing_wald = pd.concat(
    [result["wald"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_wald = bh_adjust_by(mixing_wald, ["domain", "model_set", "predictor_set"])
mixing_or = pd.concat(
    [result["odds"] for result in mixing_model_results.values()],
    ignore_index=True,
)
mixing_fit_stats = pd.concat(
    [result["fit_stats"] for result in mixing_model_results.values()],
    ignore_index=True,
)

display(
    mixing_wald[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "term",
            "chi2",
            "df",
            "P>chi2",
            "p_adj_bh",
            "n_model_rows",
            "n_nodes",
            "dropped_nonvarying_rows",
            "dropped_nonvarying_strata",
            "dropped_nonvarying_detail",
        ]
    ]
)

display(
    mixing_fit_stats[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "r2_mcfadden",
            "converged",
            "aic",
            "bic_llf",
            "log_likelihood",
            "ll_null",
            "n_model_rows",
            "n_nodes",
        ]
    ]
)

,domain,model_set,predictor_set,predictor,term,chi2,df,P>chi2,p_adj_bh,n_model_rows,n_nodes,dropped_nonvarying_rows,dropped_nonvarying_strata,dropped_nonvarying_detail
0,node_mixing,primary,single,sex_entropy_z,sex_entropy_z,1.650174,1,0.198935,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
1,node_mixing,primary,single,age_entropy_z,age_entropy_z,8.026412,1,0.004610,0.023050,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
2,node_mixing,primary,single,simd_entropy_z,simd_entropy_z,2.020922,1,0.155145,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
3,node_mixing,primary,single,urban_rural_entropy_z,urban_rural_entropy_z,0.004173,1,0.948495,0.948495,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
4,node_mixing,primary,single,health_board_entropy_z,health_board_entropy_z,1.951550,1,0.162420,0.248669,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
5,node_mixing,primary,joint,all_mixing,sex_entropy_z,0.808100,1,0.368683,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
6,node_mixing,primary,joint,all_mixing,age_entropy_z,5.699678,1,0.016968,0.084840,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
7,node_mixing,primary,joint,all_mixing,simd_entropy_z,0.369690,1,0.543173,0.543173,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
8,node_mixing,primary,joint,all_mixing,urban_rural_entropy_z,0.726623,1,0.393980,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."
9,node_mixing,primary,joint,all_mixing,health_board_entropy_z,0.757914,1,0.383982,0.492475,12966,12966,1,1,"{'rows': {'window_idx': 1}, 'strata': {'window..."


,domain,model_set,predictor_set,predictor,r2_mcfadden,converged,aic,bic_llf,log_likelihood,ll_null,n_model_rows,n_nodes
0,node_mixing,primary,single,sex_entropy_z,NaN,True,17084.730562,NaN,-8455.365281,NaN,12966,12966
1,node_mixing,primary,single,age_entropy_z,NaN,True,17078.264046,NaN,-8452.132023,NaN,12966,12966
2,node_mixing,primary,single,simd_entropy_z,NaN,True,17084.434226,NaN,-8455.217113,NaN,12966,12966
3,node_mixing,primary,single,urban_rural_entropy_z,NaN,True,17086.487631,NaN,-8456.243816,NaN,12966,12966
4,node_mixing,primary,single,health_board_entropy_z,NaN,True,17084.514233,NaN,-8455.257117,NaN,12966,12966
5,node_mixing,primary,joint,all_mixing,NaN,True,17083.659581,NaN,-8450.829790,NaN,12966,12966
6,node_mixing,expanded,single,sex_entropy_z,NaN,True,16975.707451,NaN,-8396.853726,NaN,12966,12966
7,node_mixing,expanded,single,age_entropy_z,NaN,True,16964.890580,NaN,-8391.445290,NaN,12966,12966
8,node_mixing,expanded,single,simd_entropy_z,NaN,True,16971.885459,NaN,-8394.942729,NaN,12966,12966
9,node_mixing,expanded,single,urban_rural_entropy_z,NaN,True,16974.301017,NaN,-8396.150508,NaN,12966,12966


### Mixing Odds Ratios

For entropy z-score models, odds ratios are per one-unit increase in the entropy z-score. OR < 1 means candidate nodes are less likely at higher-than-null mixing; OR > 1 means candidate nodes are more likely at higher-than-null mixing.

In [32]:
display(
    mixing_or[
        [
            "domain",
            "model_set",
            "predictor_set",
            "predictor",
            "term",
            "estimate",
            "std_error",
            "p_value",
            "odds_ratio",
            "or_low",
            "or_high",
        ]
    ].sort_values(["model_set", "predictor_set", "predictor", "p_value"])
)

,domain,model_set,predictor_set,predictor,term,estimate,std_error,p_value,odds_ratio,or_low,or_high
18,node_mixing,expanded,joint,all_mixing,urban_rural_entropy_z,0.043919,0.013156,0.000842,1.044898,1.018300,1.072191
16,node_mixing,expanded,joint,all_mixing,age_entropy_z,-0.031856,0.011132,0.004216,0.968646,0.947740,0.990013
19,node_mixing,expanded,joint,all_mixing,health_board_entropy_z,-0.019971,0.007858,0.011034,0.980227,0.965247,0.995440
17,node_mixing,expanded,joint,all_mixing,simd_entropy_z,-0.014325,0.010564,0.175097,0.985777,0.965577,1.006400
15,node_mixing,expanded,joint,all_mixing,sex_entropy_z,-0.005901,0.012732,0.643014,0.994116,0.969616,1.019236
11,node_mixing,expanded,single,age_entropy_z,age_entropy_z,-0.036345,0.010662,0.000653,0.964307,0.944364,0.984671
14,node_mixing,expanded,single,health_board_entropy_z,health_board_entropy_z,-0.015686,0.006682,0.018893,0.984436,0.971628,0.997413
10,node_mixing,expanded,single,sex_entropy_z,sex_entropy_z,-0.012839,0.012518,0.305057,0.987243,0.963315,1.011765
12,node_mixing,expanded,single,simd_entropy_z,simd_entropy_z,-0.021473,0.009757,0.027751,0.978756,0.960217,0.997653
13,node_mixing,expanded,single,urban_rural_entropy_z,urban_rural_entropy_z,0.018250,0.011532,0.113532,1.018418,0.995657,1.041699


## Interpretation Guide

Use the tables in this order:

1. **Single-predictor primary models**: confirmatory family for each socio-geodemographic variable, Firth-penalised, adjusted for window and variant context.
2. **Single-predictor expanded models**: sensitivity to local data-zone surveillance and epidemic-burden adjustment.
3. **Joint primary models**: whether each predictor retains signal when the socio-geodemographic predictors are mutually adjusted.
4. **Joint expanded models**: the most conditional specification; useful as a robustness check rather than the simplest effect summary.
5. **McFadden pseudo-R2 / likelihood summaries**: compare models within the same outcome/family and analysis frame when available. Firth fits use penalised estimation for coefficients; likelihood summaries are reported as diagnostics rather than confirmatory tests.

Composition models describe **who/where the sequences in candidate nodes come from**. Node-level mixing models describe **whether candidate nodes are unusually internally diverse or concentrated** for their size and window.


In [33]:
summary_tables = {
    "composition_wald.csv": composition_wald,
    "composition_odds_ratios.csv": composition_or,
    "composition_fit_stats.csv": composition_fit_stats,
    "mixing_wald.csv": mixing_wald,
    "mixing_odds_ratios.csv": mixing_or,
    "mixing_fit_stats.csv": mixing_fit_stats,
}


def clean_export_table(table: pd.DataFrame) -> pd.DataFrame:
    out = table.copy()
    out.columns = [str(col).strip() for col in out.columns]
    for col in out.select_dtypes(include=["object", "string"]).columns:
        present = out[col].notna()
        out.loc[present, col] = out.loc[present, col].astype(str).str.strip()
    return out


for filename, table in summary_tables.items():
    path = RESULT_DIR / filename
    clean_export_table(table).to_csv(path, index=False)
    print(f"saved {filename}: {len(table):,} rows")

saved composition_wald.csv: 20 rows
saved composition_odds_ratios.csv: 152 rows
saved composition_fit_stats.csv: 12 rows
saved mixing_wald.csv: 20 rows
saved mixing_odds_ratios.csv: 20 rows
saved mixing_fit_stats.csv: 12 rows
